# Notebook 6c: two checks before the 5 Oct checkpoint

**Runtime:** CPU is enough (no GPU needed). Run time about 5–10 minutes. It only reads files that 6a and 6b already saved to Drive, plus the data parquets.

**Part A: is the NDCG@5 gap in 6a real?** In 6a the standard fine-tuned model scored NDCG@5 0.857 and the strict one 0.781 (170 jobs), but that gap was never tested for significance. This part runs paired tests per job (bootstrap CI + Wilcoxon, Holm-corrected).

**Part B: is "tech words → Good" real fit or an artifact?** 6b found that, within the 91 jobs that hand out both Good and Potential, a classifier on the **resume text alone** predicts Good vs Potential (per-job AUC 0.635), and its top Good words are tech terms. If those 91 jobs are mostly **tech jobs**, a tech resume being "Good" for a tech job is simply real fit, and the artifact claim weakens. This part checks:

1. **Job mix.** How many of the mixed jobs are tech jobs, compared with the all-Good / all-Potential jobs?
2. **The decisive split.** Inside **non-tech** mixed jobs, does a tech-heavy resume still get "Good"? It can't be real fit there, so a signal in that group supports the artifact.
3. **Tech words removed.** If the tech vocabulary is deleted from every resume, does the resume-text-only classifier still work?

Just **Run all**. At the end, download the .ipynb (File → Download .ipynb) and send me the path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
SEED = 42

def holm(p):
    p = np.asarray(p, dtype=float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

## Part A: significance of the NDCG@5 gaps (6a)

The per-job NDCG@5 is recomputed exactly as in 6a (same tie-break seed), so the means printed first should match the 6a table: standard 0.857, size-matched 0.846, strict 0.781, TF-IDF 0.736. If they don't match, stop and tell me.

In [ ]:
ev = pd.read_parquet(f'{RESULTS}/06a_cv_pair_scores.parquet')

def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k]); return dcg(r[:k]) / ideal if ideal > 0 else 0.0

def per_job_ndcg(df, col, k=5):
    # identical to 6a's per_job: one rng for the whole pass, drawn only for eligible jobs
    r = np.random.default_rng(SEED); out = {}
    for jid, g in df.groupby('job_id'):
        rel = g.relevance.to_numpy(); s = g[col].to_numpy()
        if len(g) >= 3 and len(set(rel)) >= 2:
            out[jid] = ndcg_at_k(rel[np.lexsort((r.random(len(s)), -s))].tolist(), k)
    return pd.Series(out)

METHODS = ['FT standard', 'FT size-matched', 'FT strict', 'BGE-base (zero-shot)', 'E5-base (zero-shot)', 'TF-IDF']
nd = pd.DataFrame({m: per_job_ndcg(ev, m) for m in METHODS})
print(f'{len(nd)} jobs with NDCG@5')
display(nd.mean().round(3).to_frame('mean NDCG@5 (check against 6a)'))

In [ ]:
job_fold = ev.drop_duplicates('job_id').set_index('job_id').fold

def paired_ndcg(a, b, n=5000):
    d = (nd[a] - nd[b]).to_numpy()
    r = np.random.default_rng(SEED)
    boots = np.array([r.choice(d, len(d)).mean() for _ in range(n)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p_boot = min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))
    p_wil = wilcoxon(d, zero_method='zsplit').pvalue if np.any(d != 0) else 1.0
    folds = (nd[a] - nd[b]).groupby(job_fold).mean()
    return {'comparison': f'{a}  minus  {b}', 'mean diff': round(d.mean(), 3), 'CI': f'[{lo:.3f}, {hi:.3f}]',
            'jobs better / worse / tied': f'{(d > 0).sum()} / {(d < 0).sum()} / {(d == 0).sum()}',
            'folds where a > b': f'{(folds > 0).sum()} of {len(folds)}',
            'p bootstrap': p_boot, 'p Wilcoxon': p_wil}

PAIRS = [('FT standard', 'FT strict'),            # total gain from seeing the test resumes
         ('FT size-matched', 'FT strict'),        # memorisation at equal training size
         ('FT standard', 'FT size-matched'),      # gain from more data alone
         ('FT strict', 'BGE-base (zero-shot)'),   # does honest fine-tuning still help?
         ('FT strict', 'E5-base (zero-shot)'),
         ('FT strict', 'TF-IDF')]
ndcg_tests = pd.DataFrame([paired_ndcg(a, b) for a, b in PAIRS])
ndcg_tests['p Holm (Wilcoxon)'] = holm(ndcg_tests['p Wilcoxon'])
ndcg_tests['p Holm (bootstrap)'] = holm(ndcg_tests['p bootstrap'])
for c in ['p bootstrap', 'p Wilcoxon', 'p Holm (Wilcoxon)', 'p Holm (bootstrap)']:
    ndcg_tests[c] = ndcg_tests[c].map(lambda v: float(f'{v:.3g}'))
display(ndcg_tests)

## Part B: are the mixed jobs mostly tech jobs?

A job or resume counts **tech terms**: the number of *different* terms from a fixed tech vocabulary (programming languages, databases, web, cloud, IT support…) found in its text. A job is called **tech** if it mentions at least 3 different terms. Because that cut-off is a choice, the job-mix table also shows 2 and 5.

The list of all mixed jobs with their first line (usually the title) is saved to Drive as 06c_mixed_jobs.csv, so we can check the tech flag by eye.

In [ ]:
all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet') for s in ('train', 'val', 'test')], ignore_index=True)
fit = all_df[all_df.relevance > 0].copy().reset_index(drop=True)
fit['good'] = (fit.relevance == 2).astype(int)

TECH = {
    'software': r'\bsoftware\b', 'developer': r'\bdevelopers?\b', 'programming': r'\bprogramm(?:er|ers|ing)\b',
    'javascript': r'\bjavascript\b', 'typescript': r'\btypescript\b', 'python': r'\bpython\b', 'java': r'\bjava\b',
    'sql': r'\bsql\b', 'mysql': r'\bmysql\b', 'postgres': r'\bpostgre(?:s|sql)\b', 'mongodb': r'\bmongo(?:db)?\b',
    'html': r'\bhtml5?\b', 'css': r'\bcss3?\b', 'php': r'\bphp\b', '.net': r'(?:asp)?\.net\b', 'c#': r'\bc#',
    'c++': r'\bc\+\+', 'react': r'\breact(?:\.?js)?\b', 'angular': r'\bangular(?:js)?\b', 'node': r'\bnode\.?js\b',
    'django': r'\bdjango\b', 'aws': r'\baws\b', 'azure': r'\bazure\b', 'cloud': r'\bcloud\b', 'devops': r'\bdevops\b',
    'docker': r'\bdocker\b', 'kubernetes': r'\bkubernetes\b', 'linux': r'\blinux\b', 'database': r'\bdatabases?\b',
    'data science/analytics': r'\bdata (?:scien\w*|analy\w*|engineer\w*|warehous\w*)',
    'machine learning': r'\bmachine learning\b|\bdeep learning\b', 'network admin': r'\bnetwork (?:admin\w*|engineer\w*|security)\b',
    'cybersecurity': r'\bcyber ?security\b', 'it support': r'\bit (?:support|specialist|infrastructure)\b|\bhelp ?desk\b',
    'information technology': r'\binformation technology\b', 'full stack': r'\bfull[- ]?stack\b',
    'front end': r'\bfront[- ]?end\b', 'back end': r'\bback[- ]?end\b', 'web': r'\bweb (?:develop\w*|design\w*|applications?)\b',
    'api': r'\b(?:rest(?:ful)? )?apis?\b', 'git': r'\bgit(?:hub)?\b', 'qa/testing': r'\bselenium\b|\bqa (?:engineer|tester|analyst)\b',
    'bi tools': r'\btableau\b|\bpower ?bi\b', 'sql server': r'\bsql server\b',
}
TECH_RE = {k: re.compile(v, re.I) for k, v in TECH.items()}
def tech_count(t): return sum(1 for rx in TECH_RE.values() if rx.search(t))
def strip_tech(t):
    for rx in TECH_RE.values(): t = rx.sub(' ', t)
    return t

jobs = all_df.drop_duplicates('job_id').set_index('job_id').job_text
job_rate = fit.groupby('job_id').good.mean()
job_tab = pd.DataFrame({'good share': job_rate, 'fit pairs': fit.groupby('job_id').size()})
job_tab['group'] = np.select([job_rate == 1, job_rate == 0], ['all-Good', 'all-Potential'], 'mixed')
job_tab['tech terms'] = jobs.reindex(job_tab.index).map(tech_count)
job_tab['first line'] = jobs.reindex(job_tab.index).map(lambda t: re.sub(r'\s+', ' ', t.strip().split('\n')[0])[:90])

mix = pd.DataFrame({f'tech share (>= {c} terms)': job_tab.groupby('group')['tech terms'].apply(lambda s, c=c: round((s >= c).mean(), 3))
                    for c in (2, 3, 5)})
mix.insert(0, 'jobs', job_tab.group.value_counts())
mix['median tech terms'] = job_tab.groupby('group')['tech terms'].median()
display(mix)

from scipy.stats import fisher_exact
job_tab['tech job'] = job_tab['tech terms'] >= 3
tbl = pd.crosstab(job_tab.group == 'mixed', job_tab['tech job'])
print('Fisher exact, mixed vs pure jobs x tech vs non-tech: odds ratio %.2f, p = %.3g' % fisher_exact(tbl.values))

mixed_jobs = job_tab[job_tab.group == 'mixed'].sort_values('tech terms', ascending=False)
mixed_jobs.to_csv(f'{RESULTS}/06c_mixed_jobs.csv')
pd.set_option('display.max_colwidth', 90)
display(mixed_jobs[['tech terms', 'tech job', 'good share', 'fit pairs', 'first line']])

## Part B2: inside the mixed jobs, split by tech vs non-tech

Four scorers, per-job AUC for Good vs Potential, with bootstrap CIs over jobs (0.5 = chance):

- **Resume tech terms**: just the number of tech terms in the resume. In a *non-tech* job this can't be real fit, so AUC above 0.5 there means the label rewards tech resumes regardless of the job.
- **Resume text only (unseen resumes)**: the 6b classifier, re-trained here (grouped by resume, so each resume is scored by a model that never saw it).
- **Resume text only, tech words removed**: the same classifier after deleting every tech term from the resumes.
- **TF-IDF job–resume similarity**: a content method that *does* read the job, for comparison.

In [ ]:
def grouped_oof(df, texts, group_col):
    y = df.good.to_numpy(); oof = np.zeros(len(df)); texts = np.asarray(texts, dtype=object)
    for tr, te in GroupKFold(5).split(df, y, df[group_col]):
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50_000)
        Xtr = vec.fit_transform(texts[tr]); Xte = vec.transform(texts[te])
        clf = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced').fit(Xtr, y[tr])
        oof[te] = clf.predict_proba(Xte)[:, 1]
    return oof

res_text = all_df.drop_duplicates('resume_id').set_index('resume_id').resume_text
res_tech = res_text.map(tech_count)
fit['resume tech terms'] = fit.resume_id.map(res_tech).to_numpy()
fit['resume text only'] = grouped_oof(fit, fit.resume_text, 'resume_id')
fit['resume text only, tech words removed'] = grouped_oof(fit, fit.resume_text.map(strip_tech), 'resume_id')

tf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tf.fit(list(jobs) + list(res_text))
fit['TF-IDF job-resume similarity'] = np.asarray(tf.transform(fit.job_text).multiply(tf.transform(fit.resume_text)).sum(axis=1)).ravel()

SCORERS = ['resume tech terms', 'resume text only', 'resume text only, tech words removed', 'TF-IDF job-resume similarity']

def per_job_auc(df, col):
    out = {}
    for jid, g in df.groupby('job_id'):
        if 0 < g.good.sum() < len(g) and g[col].nunique() > 1:
            out[jid] = roc_auc_score(g.good, g[col])
        elif 0 < g.good.sum() < len(g):
            out[jid] = 0.5          # scorer gives everyone the same value: a tie is chance
    return pd.Series(out)

def boot_ci(v, n=2000):
    r = np.random.default_rng(SEED); v = np.asarray(v)
    b = np.array([r.choice(v, len(v)).mean() for _ in range(n)])
    return np.percentile(b, [2.5, 97.5]), min(1.0, 2 * min((b <= 0.5).mean(), (b >= 0.5).mean()))

mixed_ids = job_tab.index[job_tab.group == 'mixed']
tech_ids = set(job_tab.index[job_tab['tech job']])
rows = []
for col in SCORERS:
    a = per_job_auc(fit[fit.job_id.isin(mixed_ids)], col)
    for grp, ids in [('all mixed jobs', a.index), ('tech mixed jobs', [j for j in a.index if j in tech_ids]),
                     ('NON-tech mixed jobs', [j for j in a.index if j not in tech_ids])]:
        v = a.loc[list(ids)]
        if len(v) < 5:
            rows.append({'scorer': col, 'jobs': grp, 'n jobs': len(v)}); continue
        (lo, hi), p = boot_ci(v)
        rows.append({'scorer': col, 'jobs': grp, 'n jobs': len(v), 'per-job AUC': round(v.mean(), 3),
                     'CI': f'[{lo:.3f}, {hi:.3f}]', 'p vs 0.5': float(f'{p:.3g}')})
split_tab = pd.DataFrame(rows)
display(split_tab)

## Part B3: the threshold doesn't decide it

The same NON-tech test with the tech-job cut-off at 2 and at 5 terms, for the two scorers that matter most.

In [ ]:
sens = []
for cut in (2, 3, 5):
    non_tech = set(job_tab.index[(job_tab.group == 'mixed') & (job_tab['tech terms'] < cut)])
    for col in ['resume tech terms', 'resume text only', 'resume text only, tech words removed']:
        v = per_job_auc(fit[fit.job_id.isin(non_tech)], col)
        if len(v) < 5:
            sens.append({'cut-off': cut, 'scorer': col, 'n non-tech mixed jobs': len(v)}); continue
        (lo, hi), p = boot_ci(v)
        sens.append({'cut-off': cut, 'scorer': col, 'n non-tech mixed jobs': len(v), 'per-job AUC': round(v.mean(), 3),
                     'CI': f'[{lo:.3f}, {hi:.3f}]', 'p vs 0.5': float(f'{p:.3g}')})
sens_tab = pd.DataFrame(sens)
display(sens_tab)

In [ ]:
ndcg_tests.to_csv(f'{RESULTS}/06c_ndcg5_tests.csv', index=False)
mix.to_csv(f'{RESULTS}/06c_job_mix.csv')
split_tab.to_csv(f'{RESULTS}/06c_tech_split_auc.csv', index=False)
sens_tab.to_csv(f'{RESULTS}/06c_tech_cutoff_sensitivity.csv', index=False)
print('saved')